<a href="https://colab.research.google.com/github/saipravalika1224/Data-Cleaning-task/blob/main/Used_Car_Data_Prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# --- Task 1: Load, Inspect, and Clean ---

# Load dataset
url = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/used_cars.csv"
try:
    df = pd.read_csv(url)
except:
    import kagglehub
    path = kagglehub.dataset_download("manishkr1754/cardekho-used-car-data")
    # Assuming the first csv in the path
    import os
    df = pd.read_csv(os.path.join(path, os.listdir(path)[0]))

# Inspect initial data
print("Initial Shape:", df.shape)
print("\n--- Info ---")
df.info()
print("\n--- Null Percentages ---")
print(df.isnull().sum() * 100 / len(df))

# Drop rows where selling_price is missing
df.dropna(subset=['selling_price'], inplace=True)

# Clean mileage, engine, and max_power
cols_to_clean = ['mileage', 'engine', 'max_power']
for col in cols_to_clean:
    # Ensure column is string, strip units, and convert
    df[col] = df[col].astype(str).str.extract('(\d+\.?\d*)').astype(float)
    # Fill nulls with median
    df[col] = df[col].fillna(df[col].median())

# Filter selling_price
df = df[(df['selling_price'] < 999999999) & (df['selling_price'] >= 10000)]

# Drop duplicates
df.drop_duplicates(inplace=True)

print(f"\nFinal Shape after cleaning: {df.shape}")



<>:32: SyntaxWarning: invalid escape sequence '\d'
<>:32: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_8526/1594735239.py:32: SyntaxWarning: invalid escape sequence '\d'
  df[col] = df[col].astype(str).str.extract('(\d+\.?\d*)').astype(float)


Using Colab cache for faster access to the 'cardekho-used-car-data' dataset.
Initial Shape: (15411, 14)

--- Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         15411 non-null  int64  
 1   car_name           15411 non-null  object 
 2   brand              15411 non-null  object 
 3   model              15411 non-null  object 
 4   vehicle_age        15411 non-null  int64  
 5   km_driven          15411 non-null  int64  
 6   seller_type        15411 non-null  object 
 7   fuel_type          15411 non-null  object 
 8   transmission_type  15411 non-null  object 
 9   mileage            15411 non-null  float64
 10  engine             15411 non-null  int64  
 11  max_power          15411 non-null  float64
 12  seats              15411 non-null  int64  
 13  selling_price      15411 non-null  int64  
dtype

Task 1: Load, Inspect, and Clean
The data was loaded from the provided source. Cleaning involved stripping units from numeric features, handling missing values via median imputation, and filtering out price outliers and duplicate entries to ensure data quality.

In [6]:
# --- Task 2: Encode Categorical Features ---

# Label Encoding for transmission_type (using map to handle potential variations)
# Adjusting column name to 'transmission' as per standard CarDekho dataset naming
trans_col = 'transmission' if 'transmission' in df.columns else 'transmission_type'
df[trans_col] = df[trans_col].map({'Manual': 0, 'Automatic': 1})

# One-hot encoding
df = pd.get_dummies(df, columns=['fuel_type', 'seller_type'], drop_first=True)

print("\nFinal Column List:")
print(df.columns.tolist())




Final Column List:
['Unnamed: 0', 'car_name', 'brand', 'model', 'vehicle_age', 'km_driven', 'transmission_type', 'mileage', 'engine', 'max_power', 'seats', 'selling_price', 'fuel_type_Diesel', 'fuel_type_Electric', 'fuel_type_LPG', 'fuel_type_Petrol', 'seller_type_Individual', 'seller_type_Trustmark Dealer']


Task 2: Encode Categorical Features
In this step, we converted text-based categories into numerical formats that a machine learning model can process.
Why is drop_first=True used in one-hot encoding?
We use drop_first=True to avoid the Dummy Variable Trap, which leads to multicollinearity. Multicollinearity is a phenomenon where one feature can be perfectly predicted from others. For example, if we have two columns, Fuel_Type_Petrol and Fuel_Type_Diesel, and a car is not Petrol, we know with 100% certainty it is Diesel. By dropping the first column, we ensure the remaining variables are independent, which is a mathematical requirement for many models like Linear Regression.
What does a row of all zeros represent?
A row of all zeros in the dummy columns represents the Reference Category (the specific category that was dropped). For instance, if the fuel_type_Diesel column was dropped and a row has 0 for all other fuel types, it signifies that the vehicle is a Diesel car.

In [7]:
# --- Task 3: Split and Compute Baseline MAE ---

# Define X and y
y = df['selling_price']
X = df.drop(columns=['selling_price'])

# Drop non-numeric columns (like 'name' or original categorical strings)
X = X.select_dtypes(include=['number'])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Baseline Prediction: Mean of y_train
y_pred_baseline = [y_train.mean()] * len(y_test)
baseline_mae = mean_absolute_error(y_test, y_pred_baseline)

print(f"\nBaseline MAE: ₹{round(baseline_mae)}")


Baseline MAE: ₹468748


Task 3: Split and Compute Baseline MAE
The dataset was split into training (80%) and testing (20%) sets. A baseline was established by calculating the Mean Absolute Error (MAE) using the average selling price of the training set as a constant prediction for the test set.